# Strategy 2 - Cross-sectional country ETF momentum

Dollar-neutral long/short country ETF momentum overlay inspired by Asness, Moskowitz, and Pedersen, *Value and Momentum Everywhere* (JoF 2013).

Signal: rank country ETFs by 12-1 month total return. Long top 5, short bottom 5, equal-weighted by default. Rebalance weekly on Wednesday close. Names whose rank changed by less than 2 places keep the prior target weight to suppress turnover.

The strategy is intended as a market-neutral overlay: gross exposure is 100%, net exposure is approximately 0%, and benchmark correlation should be measured against SPY/EFA/EEM rather than treated as a beta replacement.

Bloomberg dispersion filter is not included here. If MXEF member realized cross-sectional dispersion is available later, add it as an external risk filter before `run_backtest` rather than mixing vendor-specific logic into the core Yahoo-based strategy.

In [ ]:
import pandas as pd

from alpha_lab.backtest.country_momentum import country_momentum_signal, country_momentum_weights
from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly
from alpha_lab.utils.paths import CONFIGS_DIR

In [ ]:
START = "2010-01-01"
END = None

LOOKBACK_MONTHS = 12
SKIP_MONTHS = 1
TOP_N = 5
BOTTOM_N = 5
REBALANCE = "W-WED"
RANK_CHANGE_THRESHOLD = 2

# "equal" is the base case. Use "inverse_vol" for volatility-weighted legs.
WEIGHTING = "equal"
VOL_LOOKBACK_DAYS = 60
LEG_GROSS = 0.5

# About 20 bps round-trip: 10 bps one-way total cost assumption.
COSTS_BPS = 5.0
SLIPPAGE_BPS = 5.0

BENCHMARKS = ["SPY", "EFA", "EEM"]

In [ ]:
universe = pd.read_csv(CONFIGS_DIR / "country_etf_universe.csv")
tickers = universe["ticker"].tolist()
universe

In [ ]:
all_prices = load_prices(tickers + BENCHMARKS, start=START, end=END)
prices = all_prices[tickers]
benchmark_prices = all_prices[BENCHMARKS]
prices.tail()

In [ ]:
momentum = country_momentum_signal(
    prices,
    lookback_months=LOOKBACK_MONTHS,
    skip_months=SKIP_MONTHS,
)
weights = country_momentum_weights(
    prices,
    lookback_months=LOOKBACK_MONTHS,
    skip_months=SKIP_MONTHS,
    top_n=TOP_N,
    bottom_n=BOTTOM_N,
    rebalance=REBALANCE,
    rank_change_threshold=RANK_CHANGE_THRESHOLD,
    leg_gross=LEG_GROSS,
    weighting=WEIGHTING,
    vol_lookback_days=VOL_LOOKBACK_DAYS,
)
weights.tail()

In [ ]:
res = run_backtest(
    weights,
    prices,
    rebalance=REBALANCE,
    costs_bps=COSTS_BPS,
    slippage_bps=SLIPPAGE_BPS,
)
bench_returns = benchmark_prices.pct_change().reindex(res.returns.index).fillna(0.0)

first_invested = res.weights.abs().sum(axis=1).gt(0).idxmax()
strategy_returns = res.returns.loc[first_invested:]
bench_returns = bench_returns.loc[first_invested:]

pd.Series(summary(strategy_returns), name="country momentum")

In [ ]:
diagnostics = pd.Series(
    {
        "start": strategy_returns.index[0].date(),
        "end": strategy_returns.index[-1].date(),
        "avg_long_gross": res.weights.loc[strategy_returns.index].clip(lower=0).sum(axis=1).mean(),
        "avg_short_gross": -res.weights.loc[strategy_returns.index].clip(upper=0).sum(axis=1).mean(),
        "avg_net_exposure": res.weights.loc[strategy_returns.index].sum(axis=1).mean(),
        "avg_one_way_turnover": res.turnover.loc[strategy_returns.index].mean(),
        "corr_SPY": strategy_returns.corr(bench_returns["SPY"]),
        "corr_EFA": strategy_returns.corr(bench_returns["EFA"]),
        "corr_EEM": strategy_returns.corr(bench_returns["EEM"]),
    }
)
diagnostics

In [ ]:
equity_curve(strategy_returns, name="country momentum")

In [ ]:
drawdown_chart(strategy_returns)

In [ ]:
heatmap_monthly(strategy_returns)

In [ ]:
latest_signal = momentum.reindex(prices.index).ffill().iloc[-1]
latest = pd.concat(
    [latest_signal.rename("momentum"), weights.iloc[-1].rename("target_weight")],
    axis=1,
).join(universe.set_index("ticker")[["region", "country"]])
latest.sort_values("momentum", ascending=False)